<a href="https://colab.research.google.com/github/ifigeneiamanolou/MultimodalAvatar/blob/Avatar/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install faster-whisper openai edge-tts python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.4 MB/s eta 0:00:00


In [12]:
from google.colab import userdata
import os
from openai import OpenAI
from dotenv import load_dotenv
import edge_tts
import asyncio
from faster_whisper import WhisperModel

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key = OPENAI_API_KEY)


In [14]:
def generate_response(user_input):
    response = client.responses.create(
        model = "gpt-5.5-nano",
        input = user_input
    )
    return response.output_text

def start_chatbot():
    print("Welcome to the avatar, say exit to stop.\n")

    with open("file.txt", "r") as file:
        user_input = file.read()

    if user_input.lower() == "exit":
        print("End of the conversation.")
        return None
    else:
        print(f"You : {user_input} \n")
        return generate_response(user_input)


async def main():
    # ASR
    model = WhisperModel(model_size_or_path="large-v3", device="cuda", compute_type="int8_float16") # To investigate models
    audio_file = "/content/drive/MyDrive/Recording.m4a"
    segments, _ = model.transcribe(
        audio_file,
        beam_size=5,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=500),
    )
    segments = list(segments)
    for seg in segments:
        print(f"for segment {seg} text is {seg.text} \n")
    print("Finished ASR \n")
    # To be moved to a Google Collab GPU and to enable parallel processing (now 3.2s)

    # Directory with the input audio file
    file = open("file.txt", "w")
    full_text = " ".join(seg.text for seg in segments)
    file.write(full_text)
    file.close()
    print("Finished text uploading \n")

    # NLP
    response = start_chatbot()
    if response:
        print(f"Bot : {response} \n")
    print("Finished NLP \n")

    # Directory with output text file
    file = open("output.txt", "w")
    file.write(response)
    file.close()
    print("Finished output text uploading \n")

    # TTS
    tts = edge_tts.Communicate(response, voice="en-US-AriaNeural")

    # Save speech in an mp4 file
    await tts.save("output.mp4")
    print("Finished audio response generation \n")

    # Generate facial animation using OVR Lip Syncing (NVIDIA)

await main()

for segment Segment(id=1, seek=0, start=2.48, end=4.48, text=' New recording', tokens=[50365, 1873, 6613, 50465], avg_logprob=-0.18876953125, compression_ratio=0.6190476190476191, no_speech_prob=0.51123046875, words=None, temperature=0.0) text is  New recording 

Welcome to the avatar, say exit to stop.

You :  New recording 



AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************64a4. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}